In [ ]:
# ======================
# Install + Imports
# ======================
!pip install opencv-python tqdm
!pip install mediapipe==0.10.13
from pathlib import Path
import pickle, cv2, numpy as np
from tqdm import tqdm
import mediapipe as mp
from google.colab import drive


In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ======================
# Paths
# ======================
DATA_DIR = Path('/content/drive/MyDrive/colab_shared/data/Df')
OUT_PKL = Path('/content/drive/MyDrive/colab_shared/outputs/DBF_reps.pkl')

LABEL_MAP = {'correct': 1, 'incorrect': 0}

In [ ]:
!ls "/content/drive/MyDrive/colab_shared/data/Df/correct"
!ls "/content/drive/MyDrive/colab_shared/data/Df/incorrect"

20260116_154642.mp4  20260116_170852.mp4  20260116_174615.mp4
20260116_154839.mp4  20260116_171128.mp4  20260116_174758.mp4
20260116_155228.mp4  20260116_171507.mp4  20260116_174938.mp4
20260116_163712.mp4  20260116_171706.mp4  20260116_175115.mp4
20260116_163854.mp4  20260116_171922.mp4  20260116_175245.mp4
20260116_164050.mp4  20260116_172204.mp4  20260116_175449.mp4
20260116_164311.mp4  20260116_172413.mp4  20260116_175629.mp4
20260116_164509.mp4  20260116_172556.mp4  20260116_175752.mp4
20260116_164726.mp4  20260116_172736.mp4  20260116_180801.mp4
20260116_164941.mp4  20260116_172943.mp4  20260116_180950.mp4
20260116_165149.mp4  20260116_173144.mp4  20260116_181130.mp4
20260116_165440.mp4  20260116_173306.mp4  20260116_181322.mp4
20260116_165726.mp4  20260116_173454.mp4  20260116_181504.mp4
20260116_165951.mp4  20260116_173808.mp4  20260116_181749.mp4
20260116_170153.mp4  20260116_174032.mp4  20260116_181926.mp4
20260116_170429.mp4  20260116_174217.mp4  20260116_182111.mp4
20260116

In [ ]:
# ======================
# Pose Setup
# ======================
mp_pose = mp.solutions.pose

SELECTED_LANDMARKS = [
    mp_pose.PoseLandmark.RIGHT_SHOULDER,  # 0
    mp_pose.PoseLandmark.LEFT_SHOULDER,   # 1
    mp_pose.PoseLandmark.RIGHT_ELBOW,     # 2
    mp_pose.PoseLandmark.LEFT_ELBOW,      # 3
    mp_pose.PoseLandmark.RIGHT_WRIST,     # 4
    mp_pose.PoseLandmark.LEFT_WRIST       # 5
]

In [ ]:
def extract_keypoints(results):
    if results.pose_landmarks:
        return [[results.pose_landmarks.landmark[lm].x,
                 results.pose_landmarks.landmark[lm].y,
                 results.pose_landmarks.landmark[lm].z,
                 results.pose_landmarks.landmark[lm].visibility]
                for lm in SELECTED_LANDMARKS]
    else:
        return [[0]*4 for _ in SELECTED_LANDMARKS]

In [ ]:
# ======================
# 🔥 DISTANCE (MAIN SIGNAL)
# ======================
def get_normalized_wrist_distance(keypoints):
    r_wrist = np.array(keypoints[4][:2])
    l_wrist = np.array(keypoints[5][:2])

    r_shoulder = np.array(keypoints[0][:2])
    l_shoulder = np.array(keypoints[1][:2])

    wrist_dist = np.linalg.norm(r_wrist - l_wrist)
    shoulder_dist = np.linalg.norm(r_shoulder - l_shoulder)

    return wrist_dist / (shoulder_dist + 1e-6)

In [ ]:
# ======================
# Rep Extraction
# ======================
def process_video(video_path: Path, label: int):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"Cannot open {video_path}")
        return []

    reps = []
    frame_buffer = []
    direction = 0  # 0=open wait, 1=close wait, 2=recording

    # 🔥 TUNED for your setup
    CLOSE_THRESHOLD = 0.6   # arms together
    OPEN_THRESHOLD = 1.4    # arms wide
    MIN_FRAMES = 3

    with mp_pose.Pose(min_detection_confidence=0.5,
                      min_tracking_confidence=0.5) as pose:

        up_count, down_count = 0, 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = pose.process(rgb)
            keypoints = extract_keypoints(results)

            dist = get_normalized_wrist_distance(keypoints)

            # DEBUG (remove later)
            print(f"Dist: {dist:.2f}")

            # ======================
            # WAIT OPEN (start)
            # ======================
            if direction == 0 and dist > OPEN_THRESHOLD:
                down_count += 1
                if down_count >= MIN_FRAMES:
                    direction = 1
                    down_count = 0
            else:
                if direction == 0:
                    down_count = 0

            # ======================
            # DETECT CLOSE
            # ======================
            if direction == 1 and dist < CLOSE_THRESHOLD:
                up_count += 1
                if up_count >= MIN_FRAMES:
                    direction = 2
                    frame_buffer = []
                    up_count = 0
            else:
                if direction == 1:
                    up_count = 0

            # ======================
            # RECORDING
            # ======================
            if direction == 2:
                frame_buffer.append(np.array(keypoints).flatten())

            # ======================
            # END REP
            # ======================
            if direction == 2 and dist > OPEN_THRESHOLD:
                down_count += 1
                if down_count >= MIN_FRAMES:

                    if len(frame_buffer) > 10:
                        reps.append({
                            "video_id": video_path.stem,
                            "label": label,
                            "frames": np.array(frame_buffer)
                        })
                        print(f"[{video_path.stem}] Rep detected! Total reps: {len(reps)}")

                    frame_buffer = []
                    direction = 0
                    down_count = 0
            else:
                if direction == 2:
                    down_count = 0

    cap.release()
    print(f"Finished {video_path.stem} -> {len(reps)} reps")
    return reps


In [ ]:
# ======================
# Run Extraction
# ======================
def run_extraction():
    OUT_PKL.parent.mkdir(parents=True, exist_ok=True)
    all_reps = []

    for folder in ['correct','incorrect']:
        folder_path = DATA_DIR / folder

        if not folder_path.exists():
            print(f"{folder_path} not found!")
            continue

        videos = sorted(folder_path.glob('*.mp4'))

        for vid in tqdm(videos, desc=f'Processing {folder}'):
            reps = process_video(vid, LABEL_MAP[folder])
            all_reps.extend(reps)

    with open(OUT_PKL, 'wb') as f:
        pickle.dump(all_reps, f)

    print(f"Saved {len(all_reps)} reps to {OUT_PKL}")

In [ ]:
# ======================
# Check data
# ======================
run_extraction()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# ======================
# Check Data
# ======================
with open(OUT_PKL, 'rb') as f:
    data = pickle.load(f)

print(f"Total reps stored: {len(data)}\n")

for i, rep in enumerate(data[:2]):
    print(f"Rep {i+1}:")
    print(f" Label: {rep['label']} ({'Correct' if rep['label']==1 else 'Incorrect'})")
    print(f" Frames shape: {rep['frames'].shape}")
    print(f" Number of frames: {len(rep['frames'])}\n")
    # print(f" Meta: {rep['meta']}")
    # print()

Total reps stored: 1692

Rep 1:
 Label: 1 (Correct)
 Frames shape: (25, 24)
 Number of frames: 25

Rep 2:
 Label: 1 (Correct)
 Frames shape: (27, 24)
 Number of frames: 27



In [ ]:
# Load
with open(OUT_PKL, 'rb') as f:
    data = pickle.load(f)

labels = [d['label'] for d in data]
labels = np.array(labels)

total = len(labels)
correct = np.sum(labels == 1)
incorrect = np.sum(labels == 0)

print(f"Total reps: {total}")
print(f"Correct reps: {correct}")
print(f"Incorrect reps: {incorrect}")
print(f"Ratio (Correct/Incorrect): {correct}/{incorrect}")

Total reps: 1692
Correct reps: 933
Incorrect reps: 759
Ratio (Correct/Incorrect): 933/759
